
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 04 - Working with Complex Data Types in Spark

This demonstration shows how to effectively work with nested data structures in Spark, including structs, arrays, and maps, using real e-commerce data examples.

### Objectives
- Convert JSON string data to Spark SQL native complex types
- Understand and manipulate complex data types (Struct, Array, Map)
- Process nested JSON-like data structures
- Use the pivot and explode functions to reshape datasets as required

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.


## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.

Also, It will create a temp table for you named `raw_user_data`
<br></br>

```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run ./Includes/Classroom-Setup-04

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Course Catalog:,
Your Schema:,


#### Querying the newly created table

In [0]:
%sql
select * from raw_user_data

user_id,name,active,signup_date,interests,recent_purchases
101,Alice Smith,true,2023-01-15,"[""hiking"", ""machine learning"", ""photography""]","[{""product_id"": ""P123"", ""name"": ""Laptop"", ""price"": 1299.99, ""date"": ""2023-03-10""}, {""product_id"": ""P456"", ""name"": ""External Monitor"", ""price"": 249.99, ""date"": ""2023-03-15""}]"
102,Bob Johnson,true,2023-02-20,"[""coding"", ""gaming"", ""reading""]","[{""product_id"": ""P789"", ""name"": ""Mechanical Keyboard"", ""price"": 149.99, ""date"": ""2023-04-05""}]"
103,Charlie Williams,false,2022-11-05,"[""golf"", ""cooking"", ""traveling""]",[]
104,Diana Garcia,true,2023-03-10,"[""drawing"", ""yoga"", ""music""]","[{""product_id"": ""P234"", ""name"": ""Graphics Tablet"", ""price"": 199.99, ""date"": ""2023-03-25""}, {""product_id"": ""P567"", ""name"": ""Stylus Pen"", ""price"": 49.99, ""date"": ""2023-03-25""}, {""product_id"": ""P890"", ""name"": ""Design Software"", ""price"": 299.99, ""date"": ""2023-04-02""}]"
105,Ethan Davis,true,2022-09-15,"[""basketball"", ""programming"", ""movies""]","[{""product_id"": ""P321"", ""name"": ""Textbook"", ""price"": 89.99, ""date"": ""2023-01-10""}, {""product_id"": ""P654"", ""name"": ""Backpack"", ""price"": 59.99, ""date"": ""2023-01-10""}]"
106,Fiona Miller,true,2023-04-01,"[""social media"", ""writing"", ""photography""]","[{""product_id"": ""P987"", ""name"": ""Camera"", ""price"": 599.99, ""date"": ""2023-04-15""}]"
107,George Wilson,false,2022-12-10,"[""finance"", ""cycling"", ""chess""]","[{""product_id"": ""P111"", ""name"": ""Financial Software"", ""price"": 199.99, ""date"": ""2023-01-05""}, {""product_id"": ""P222"", ""name"": ""Wireless Mouse"", ""price"": 29.99, ""date"": ""2023-02-15""}]"
108,Hannah Brown,true,2023-02-28,"[""education"", ""reading"", ""gardening""]","[{""product_id"": ""P333"", ""name"": ""Educational Subscription"", ""price"": 14.99, ""date"": ""2023-03-01""}, {""product_id"": ""P444"", ""name"": ""Notebook Set"", ""price"": 24.99, ""date"": ""2023-03-01""}]"
109,Ian Taylor,true,2022-10-20,"[""cooking"", ""food"", ""travel""]","[{""product_id"": ""P555"", ""name"": ""Cooking Knives"", ""price"": 179.99, ""date"": ""2023-01-25""}, {""product_id"": ""P666"", ""name"": ""Recipe Book"", ""price"": 39.99, ""date"": ""2023-02-10""}, {""product_id"": ""P777"", ""name"": ""Spice Set"", ""price"": 49.99, ""date"": ""2023-03-20""}]"
110,Julia Martinez,true,2023-03-15,"[""law"", ""politics"", ""hiking""]","[{""product_id"": ""P888"", ""name"": ""Legal Reference Book"", ""price"": 129.99, ""date"": ""2023-04-10""}]"


## B. Convert from JSON Strings to StructTypes

Given raw data which includes nested JSON strings (arrays and/or objects), we will convert this data to native `StructTypes` in the DataFrame API.

#### Why Convert JSON Strings to StructTypes?

JSON strings in Spark DataFrames come with several inefficiencies:

1. **Parsing Overhead**: Every time you query JSON strings, Spark needs to parse them, adding computational overhead
2. **Memory Inefficiency**: JSON strings store field names repeatedly for every row, wasting memory
3. **No Type Safety**: JSON strings don't enforce data types, leading to potential errors
4. **Poor Query Performance**: Spark can't optimize queries on JSON string content as effectively
5. **Limited Predicate Pushdown**: Filter operations can't leverage columnar storage optimizations

### Steps to Convert JSON Strings to StructTypes

1. **Infer Schema**: Determine the structure of the JSON data (the `schema_of_json` function can be used for this)
2. **Apply Schema**: Use `from_json()` to convert strings to structured data
3. **Validate**: Ensure all data is correctly parsed and types are appropriate
4. **Optimize**: Once converted, optimize storage/processing if needed

### Benefits of StructTypes

1. **Columnar Storage**: Efficient storage with Parquet/Delta
2. **Type Safety**: Schema enforcement prevents data errors
3. **Query Optimization**: Spark can optimize queries better with typed data
4. **Predicate Pushdown**: Filters can be pushed down to storage layer
5. **Better Performance**: Faster queries and reduced memory usage

In [0]:
from pyspark.sql.functions import *

# Load some data which includes JSON strings
raw_user_data_df = spark.read.table("raw_user_data")

In [0]:
# Inspect the Data
raw_user_data_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- active: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- interests: string (nullable = true)
 |-- recent_purchases: string (nullable = true)



In [0]:
# Displaying the Dataframe 
display(raw_user_data_df)

user_id,name,active,signup_date,interests,recent_purchases
101,Alice Smith,true,2023-01-15,"[""hiking"", ""machine learning"", ""photography""]","[{""product_id"": ""P123"", ""name"": ""Laptop"", ""price"": 1299.99, ""date"": ""2023-03-10""}, {""product_id"": ""P456"", ""name"": ""External Monitor"", ""price"": 249.99, ""date"": ""2023-03-15""}]"
102,Bob Johnson,true,2023-02-20,"[""coding"", ""gaming"", ""reading""]","[{""product_id"": ""P789"", ""name"": ""Mechanical Keyboard"", ""price"": 149.99, ""date"": ""2023-04-05""}]"
103,Charlie Williams,false,2022-11-05,"[""golf"", ""cooking"", ""traveling""]",[]
104,Diana Garcia,true,2023-03-10,"[""drawing"", ""yoga"", ""music""]","[{""product_id"": ""P234"", ""name"": ""Graphics Tablet"", ""price"": 199.99, ""date"": ""2023-03-25""}, {""product_id"": ""P567"", ""name"": ""Stylus Pen"", ""price"": 49.99, ""date"": ""2023-03-25""}, {""product_id"": ""P890"", ""name"": ""Design Software"", ""price"": 299.99, ""date"": ""2023-04-02""}]"
105,Ethan Davis,true,2022-09-15,"[""basketball"", ""programming"", ""movies""]","[{""product_id"": ""P321"", ""name"": ""Textbook"", ""price"": 89.99, ""date"": ""2023-01-10""}, {""product_id"": ""P654"", ""name"": ""Backpack"", ""price"": 59.99, ""date"": ""2023-01-10""}]"
106,Fiona Miller,true,2023-04-01,"[""social media"", ""writing"", ""photography""]","[{""product_id"": ""P987"", ""name"": ""Camera"", ""price"": 599.99, ""date"": ""2023-04-15""}]"
107,George Wilson,false,2022-12-10,"[""finance"", ""cycling"", ""chess""]","[{""product_id"": ""P111"", ""name"": ""Financial Software"", ""price"": 199.99, ""date"": ""2023-01-05""}, {""product_id"": ""P222"", ""name"": ""Wireless Mouse"", ""price"": 29.99, ""date"": ""2023-02-15""}]"
108,Hannah Brown,true,2023-02-28,"[""education"", ""reading"", ""gardening""]","[{""product_id"": ""P333"", ""name"": ""Educational Subscription"", ""price"": 14.99, ""date"": ""2023-03-01""}, {""product_id"": ""P444"", ""name"": ""Notebook Set"", ""price"": 24.99, ""date"": ""2023-03-01""}]"
109,Ian Taylor,true,2022-10-20,"[""cooking"", ""food"", ""travel""]","[{""product_id"": ""P555"", ""name"": ""Cooking Knives"", ""price"": 179.99, ""date"": ""2023-01-25""}, {""product_id"": ""P666"", ""name"": ""Recipe Book"", ""price"": 39.99, ""date"": ""2023-02-10""}, {""product_id"": ""P777"", ""name"": ""Spice Set"", ""price"": 49.99, ""date"": ""2023-03-20""}]"
110,Julia Martinez,true,2023-03-15,"[""law"", ""politics"", ""hiking""]","[{""product_id"": ""P888"", ""name"": ""Legal Reference Book"", ""price"": 129.99, ""date"": ""2023-04-10""}]"


1. View the data above and find columns **interests** and **recent_purchases**
2. **interests** is an array column of String elements and **recent_purchases** is column containing objects

In [0]:
# Interests is an array of strings, using predefined schema
interests_schema = ArrayType(StringType())

In [0]:
# Let's get the schema for the "recent_purchases" and cast all of the columns from the raw data set into a confirmed structure

# Take a sample of one value of the "recent_purchases" column, bring this back to the Driver
recent_purchases_json = raw_user_data_df.select("recent_purchases").limit(1).collect()[0][0]
print("Raw JSON string:", recent_purchases_json)

Raw JSON string: [{"product_id": "P123", "name": "Laptop", "price": 1299.99, "date": "2023-03-10"}, {"product_id": "P456", "name": "External Monitor", "price": 249.99, "date": "2023-03-15"}]



#### Use the **schema_of_json** Function to generate a Schema based upon a sample row of data

In many cases, especially with multiple nested complex structures, it is easiest to generate a schema based upon a sample of JSON data, we can do this using the schema_of_json Function.  

From the above dataset, we can see that **interests** is an array column of string elements, **recent_purchases** is an array column which contains objects.

In [0]:
# Get the schema for the recent_purchases JSON
recent_purchases_schema = schema_of_json(lit(recent_purchases_json))

In [0]:
# Parse columns with the correct schemas
parsed_users_df = raw_user_data_df.select(
    col("user_id").cast("integer"),
    col("name"),
    col("active").cast("boolean"),
    col("signup_date").cast("date"),
    from_json(col("interests"), interests_schema).alias("interests"),
    from_json(col("recent_purchases"), recent_purchases_schema).alias("recent_purchases")
)

# Examine the schema
parsed_users_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- interests: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- recent_purchases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product_id: string (nullable = true)



In [0]:
# Now look at the data again, You will notice order has been changed for recent_purchases
display(parsed_users_df)

user_id,name,active,signup_date,interests,recent_purchases
101,Alice Smith,true,2023-01-15,"List(hiking, machine learning, photography)","List(List(2023-03-10, Laptop, 1299.99, P123), List(2023-03-15, External Monitor, 249.99, P456))"
102,Bob Johnson,true,2023-02-20,"List(coding, gaming, reading)","List(List(2023-04-05, Mechanical Keyboard, 149.99, P789))"
103,Charlie Williams,false,2022-11-05,"List(golf, cooking, traveling)",List()
104,Diana Garcia,true,2023-03-10,"List(drawing, yoga, music)","List(List(2023-03-25, Graphics Tablet, 199.99, P234), List(2023-03-25, Stylus Pen, 49.99, P567), List(2023-04-02, Design Software, 299.99, P890))"
105,Ethan Davis,true,2022-09-15,"List(basketball, programming, movies)","List(List(2023-01-10, Textbook, 89.99, P321), List(2023-01-10, Backpack, 59.99, P654))"
106,Fiona Miller,true,2023-04-01,"List(social media, writing, photography)","List(List(2023-04-15, Camera, 599.99, P987))"
107,George Wilson,false,2022-12-10,"List(finance, cycling, chess)","List(List(2023-01-05, Financial Software, 199.99, P111), List(2023-02-15, Wireless Mouse, 29.99, P222))"
108,Hannah Brown,true,2023-02-28,"List(education, reading, gardening)","List(List(2023-03-01, Educational Subscription, 14.99, P333), List(2023-03-01, Notebook Set, 24.99, P444))"
109,Ian Taylor,true,2022-10-20,"List(cooking, food, travel)","List(List(2023-01-25, Cooking Knives, 179.99, P555), List(2023-02-10, Recipe Book, 39.99, P666), List(2023-03-20, Spice Set, 49.99, P777))"
110,Julia Martinez,true,2023-03-15,"List(law, politics, hiking)","List(List(2023-04-10, Legal Reference Book, 129.99, P888))"


## C. Working with Arrays

Let's explore different ways to access and manipulate arrays.

In [0]:
# Use the array_size method to see the lengths of the array columns in the dataframe
display(
parsed_users_df.select(
    "user_id",
    array_size("interests").alias("number_of_interests"),
    array_size("recent_purchases").alias("number_of_recent_purchases")
    )
)

user_id,number_of_interests,number_of_recent_purchases
101,3,2
102,3,1
103,3,0
104,3,3
105,3,2
106,3,1
107,3,2
108,3,2
109,3,3
110,3,1


###1. The explode Method

The `explode` method is used to unnest array elements into records

In [0]:
# Let's start by simplifying the data by selecting only the columns we need
user_101s_interests_df = parsed_users_df.select("user_id", "interests").filter(parsed_users_df.user_id == 101)
display(user_101s_interests_df)

user_id,interests
101,"List(hiking, machine learning, photography)"


In [0]:
# Let's demonstrate explode, note how there are three rows associated with "user_id" 101 (one for each "interests" array element)
display(
    user_101s_interests_df.select(
        "user_id", 
        explode("interests").alias("interest")
    )    
)

user_id,interest
101,hiking
101,machine learning
101,photography


###2. The collect_set and collect_list Methods

The `collect_set` and `collect_list` methods are aggregate functions (which typically operate on grouped data) to create arrays from column values.  

`collect_list` may include duplicate values, while `collect_set` removes duplicate array elements should they exist.

In [0]:
# Let's start by creating a new DataFrame with the "interests" column exploded
exploded_df = parsed_users_df.select("user_id", explode("interests").alias("interest"))
display(exploded_df)

user_id,interest
101,hiking
101,machine learning
101,photography
102,coding
102,gaming
102,reading
103,golf
103,cooking
103,traveling
104,drawing


In [0]:
# Let's use `collect_list` to collect all the "interests" values into a list for each "user_id"
user_interests_df = exploded_df.groupBy("user_id").agg(collect_list("interest").alias("interests"))
display(user_interests_df)

user_id,interests
101,"List(hiking, machine learning, photography)"
102,"List(coding, gaming, reading)"
103,"List(golf, cooking, traveling)"
104,"List(drawing, yoga, music)"
105,"List(basketball, programming, movies)"
106,"List(social media, writing, photography)"
108,"List(education, reading, gardening)"
107,"List(finance, cycling, chess)"
109,"List(cooking, food, travel)"
110,"List(law, politics, hiking)"


## D. Referencing Struct Fields

Let's explore how to access fields within a struct (an object with a predefined schema).

In [0]:
# First let's explode the "recent_purchases" column
exploded_purchases_df = parsed_users_df.select("user_id", explode("recent_purchases").alias("purchase"))
display(exploded_purchases_df)

user_id,purchase
101,"List(2023-03-10, Laptop, 1299.99, P123)"
101,"List(2023-03-15, External Monitor, 249.99, P456)"
102,"List(2023-04-05, Mechanical Keyboard, 149.99, P789)"
104,"List(2023-03-25, Graphics Tablet, 199.99, P234)"
104,"List(2023-03-25, Stylus Pen, 49.99, P567)"
104,"List(2023-04-02, Design Software, 299.99, P890)"
105,"List(2023-01-10, Textbook, 89.99, P321)"
105,"List(2023-01-10, Backpack, 59.99, P654)"
106,"List(2023-04-15, Camera, 599.99, P987)"
107,"List(2023-01-05, Financial Software, 199.99, P111)"


In [0]:
# Use the dot notation to access struct fields
recent_purchases_df = exploded_purchases_df.select(
                        "user_id", 
                        col("purchase.date").alias("purchase_date"), 
                        col("purchase.product_id").alias("product_id"), 
                        col("purchase.name").alias("product_name"), 
                        col("purchase.price").alias("purchase_price")
                    )
display(recent_purchases_df)

user_id,purchase_date,product_id,product_name,purchase_price
101,2023-03-10,P123,Laptop,1299.99
101,2023-03-15,P456,External Monitor,249.99
102,2023-04-05,P789,Mechanical Keyboard,149.99
104,2023-03-25,P234,Graphics Tablet,199.99
104,2023-03-25,P567,Stylus Pen,49.99
104,2023-04-02,P890,Design Software,299.99
105,2023-01-10,P321,Textbook,89.99
105,2023-01-10,P654,Backpack,59.99
106,2023-04-15,P987,Camera,599.99
107,2023-01-05,P111,Financial Software,199.99


In [0]:
# We can also use the getField() method to reference columns inside of structs, here's an example....

field_access_df = exploded_purchases_df.select(
    "user_id",
    col("purchase").getField("date").alias("purchase_date"),
    col("purchase").getField("product_id").alias("product_id"),
    col("purchase").getField("name").alias("product_name"),
    col("purchase").getField("price").alias("price")
)

display(field_access_df)

user_id,purchase_date,product_id,product_name,price
101,2023-03-10,P123,Laptop,1299.99
101,2023-03-15,P456,External Monitor,249.99
102,2023-04-05,P789,Mechanical Keyboard,149.99
104,2023-03-25,P234,Graphics Tablet,199.99
104,2023-03-25,P567,Stylus Pen,49.99
104,2023-04-02,P890,Design Software,299.99
105,2023-01-10,P321,Textbook,89.99
105,2023-01-10,P654,Backpack,59.99
106,2023-04-15,P987,Camera,599.99
107,2023-01-05,P111,Financial Software,199.99


## E. Using the pivot Method

The `pivot` method in Spark allows you to transform row data into columnar format, creating a cross-tabulation. This is particularly useful for feature engineering when analyzing categorical data or when you need to reshape your data for reporting or visualization.

In [0]:
# Let's pivot the purchase data to show the count of each product purchased by each user
pivot_df = (recent_purchases_df
    .groupBy("user_id")
    .pivot("product_name")
    .agg(count("product_id").alias("quantity_purchased"))
)

# Display the result
display(pivot_df)

user_id,Backpack,Camera,Cooking Knives,Design Software,Educational Subscription,External Monitor,Financial Software,Graphics Tablet,Laptop,Legal Reference Book,Mechanical Keyboard,Notebook Set,Recipe Book,Spice Set,Stylus Pen,Textbook,Wireless Mouse
108,null,null,null,null,1,null,null,null,null,null,null,1,null,null,null,null,null
101,null,null,null,null,null,1,null,null,1,null,null,null,null,null,null,null,null
107,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,1
102,null,null,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null
109,null,null,1,null,null,null,null,null,null,null,null,null,1,1,null,null,null
105,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,null
110,null,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null
106,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
104,null,null,null,1,null,null,null,1,null,null,null,null,null,null,1,null,null


In [0]:
# Replace null values with zeros for better readability
pivot_df_no_nulls = (recent_purchases_df
    .groupBy("user_id")
    .pivot("product_name")
    .agg(count("product_id").alias("quantity_purchased"))
    .fillna(0)
)

# Display the result
display(pivot_df_no_nulls)

user_id,Backpack,Camera,Cooking Knives,Design Software,Educational Subscription,External Monitor,Financial Software,Graphics Tablet,Laptop,Legal Reference Book,Mechanical Keyboard,Notebook Set,Recipe Book,Spice Set,Stylus Pen,Textbook,Wireless Mouse
108,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0
101,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0
107,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1
102,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
109,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0
105,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
110,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
106,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
104,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0


## Key Takeaways

1. **Struct Type Operations**:
   - Use dot notation for simple access
   - `getField()` for dynamic column access
   - Maintain schema clarity

2. **Array Operations**:
   - Use array functions for manipulation
   - Leverage explode for detailed analysis
   - Consider performance with large arrays

3. **Complex Aggregate Functions**:
   - Use the `collect_list` and `collect_set` methods to create arrays from grouped data
   - Use the `pivot` function to transform row values into columns for analysis and reporting



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
